# Lock-free SPSC queue latency analysis

This notebook analyzes the `latency_throughput.csv` outputs generated by `./run_containerized_harness.sh <output_dir>`.

It provides:
- a box plot comparing latency distributions for every queue variant,
- per-variant distribution overlays comparing acquire/release vs seq_cst variants.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Set this to the host output dir passed to run_containerized_harness.sh
DATA_DIR = Path('results')

required_columns = {'PushTimestamp', 'PopTimestamp'}
rows = []

for csv_path in sorted(DATA_DIR.glob('*.csv')):
    variant = csv_path.stem
    frame = pd.read_csv(csv_path)

    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f'{csv_path} is missing columns: {sorted(missing_columns)}')

    frame['PushTimestamp'] = pd.to_numeric(frame['PushTimestamp'], errors='coerce')
    frame['PopTimestamp'] = pd.to_numeric(frame['PopTimestamp'], errors='coerce')
    frame = frame.dropna(subset=['PushTimestamp', 'PopTimestamp']).copy()

    frame['latency_us'] = (frame['PopTimestamp'] - frame['PushTimestamp']) * 1_000_000
    frame = frame[frame['latency_us'] >= 0].copy()

    frame['variant'] = variant
    frame['memory_model'] = 'seq_cst' if variant.endswith('-seq_cst') else 'acq_rel'
    frame['base_variant'] = variant.replace('-seq_cst', '')

    rows.append(frame[['variant', 'base_variant', 'memory_model', 'latency_us']])

if not rows:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR.resolve()}')

latencies = pd.concat(rows, ignore_index=True)
latencies.head()

In [ ]:
summary = latencies.groupby('variant')['latency_us'].describe(percentiles=[0.5, 0.9, 0.99])
summary

In [ ]:
preferred_base_order = [
    '0-naive',
    '1-padding_without_cached_indices',
    '2-cached_indices_without_padding',
    '1-cached_indices_and_padding',
    '2-bitwise_modulus',
]

base_variants_present = sorted(latencies['base_variant'].unique(), key=lambda v: (v not in preferred_base_order, preferred_base_order.index(v) if v in preferred_base_order else v))
variant_order = [variant for base in base_variants_present for variant in (base, f'{base}-seq_cst') if variant in latencies['variant'].values]
box_data = [latencies.loc[latencies['variant'] == variant, 'latency_us'] for variant in variant_order]

fig, ax = plt.subplots(figsize=(12, 6))
ax.boxplot(box_data, labels=variant_order, showfliers=False)
ax.set_title('Latency distribution per queue variant')
ax.set_ylabel('Latency (microseconds)')
ax.set_xlabel('Variant')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
base_variants = base_variants_present
fig, axes = plt.subplots(1, len(base_variants), figsize=(6 * len(base_variants), 4), sharey=True)

if len(base_variants) == 1:
    axes = [axes]

for ax, base_variant in zip(axes, base_variants):
    subset = latencies[latencies['base_variant'] == base_variant]

    acq_rel = subset[subset['memory_model'] == 'acq_rel']['latency_us']
    seq_cst = subset[subset['memory_model'] == 'seq_cst']['latency_us']

    if len(acq_rel) > 0:
        ax.hist(acq_rel, bins=50, density=True, alpha=0.5, label='acq_rel', color='tab:blue')
    if len(seq_cst) > 0:
        ax.hist(seq_cst, bins=50, density=True, alpha=0.5, label='seq_cst', color='tab:orange')

    ax.set_title(base_variant)
    ax.set_xlabel('Latency (microseconds)')
    ax.legend()

axes[0].set_ylabel('Density')
fig.suptitle('Latency distributions: acq_rel vs seq_cst by variant', y=1.02)
plt.tight_layout()
plt.show()